In [ ]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt

mdb_path = r"C:\Users\Jouke\Desktop\emergo_dataset\DemoDataBetsyBike.mdb"

conn_str = (
    r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};"
    fr"DBQ={mdb_path};"
)
conn = pyodbc.connect(conn_str)

In [ ]:
cursor = conn.cursor()
tables = [row.table_name for row in cursor.tables(tableType='TABLE')]
print("Gevonden tabellen:", tables)

In [ ]:
dfs = {t: pd.read_sql(f"SELECT * FROM [{t}]", conn) for t in tables}
#conn.close()

In [ ]:
dfs.keys()

In [ ]:
df_sales = dfs['SalesOrderDetail'].copy()
df_product = dfs['Product'].copy()
df_subcat = dfs['ProductSubcategory'].copy()
df_category = dfs['ProductCategory'].copy()

In [ ]:
df_sales['Revenue'] = (
    df_sales['OrderQty'].astype(float)
    * df_sales['UnitPrice'].astype(float)
    * (1 - df_sales['UnitPriceDiscount'].astype(float))
)

In [ ]:
product_dim = (
    df_product[['ProductID', 'Name', 'ProductSubcategoryID']]
    .drop_duplicates(subset=['ProductID'])
    .rename(columns={'Name': 'ProductName'})
)
subcat_dim = (
    df_subcat[['ProductSubcategoryID', 'ProductCategoryID']]
    .drop_duplicates(subset=['ProductSubcategoryID'])
)
category_dim = (
    df_category[['ProductCategoryID', 'Name']]
    .drop_duplicates(subset=['ProductCategoryID'])
    .rename(columns={'Name': 'Category'})
)

In [ ]:
sales_with_product = (
    df_sales
    .merge(product_dim, on='ProductID', how='left', validate='m:1')
    .merge(subcat_dim, on='ProductSubcategoryID', how='left', validate='m:1')
    .merge(category_dim, on='ProductCategoryID', how='left', validate='m:1')
)

In [ ]:
print("Aantal regels:", len(sales_with_product))
print("Totaalomzet: €", round(sales_with_product['Revenue'].sum(), 2))

In [ ]:
top_x = 25
top_products = (
    sales_with_product.groupby(['ProductID', 'ProductName', 'Category'], as_index=False)['Revenue']
    .sum()
    .sort_values('Revenue', ascending=False)
    .head(top_x)
)

print("\nTop 10 producten:")
print(top_products)

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(top_products['ProductName'][::-1], top_products['Revenue'][::-1])
plt.title(f"Top {top_x} producten op basis van omzet")
plt.xlabel("Omzet (€)")
plt.ylabel("Productnaam")
plt.tight_layout()
plt.show()